In [1]:
import pandas as pd
import os
from PIL import Image
import torch
from torchvision import transforms, datasets
from sklearn.model_selection import train_test_split


In [53]:
# Define paths
train_metadata_path = "/kaggle/input/isic-data/ISIC_2019_Training_Metadata.csv"
train_ground_truth_path = "/kaggle/input/isic-data/ISIC_2019_Training_GroundTruth.csv"
test_metadata_path = "/kaggle/input/isic-data/ISIC_2019_Test_Metadata.csv"
test_ground_truth_path = "/kaggle/input/isic-data/ISIC_2019_Test_GroundTruth.csv"
train_image_dir = "/kaggle/input/isic-data/ISIC_2019_Training_Input/ISIC_2019_Training_Input"
test_image_dir = "/kaggle/input/isic-data/ISIC_2019_Test_Input/ISIC_2019_Test_Input"


In [3]:
def load_and_clean_metadata(metadata_path, ground_truth_path):
    metadata = pd.read_csv(metadata_path)
    ground_truth = pd.read_csv(ground_truth_path)

    # Handle missing values
    metadata.fillna({
        'age_approx': metadata['age_approx'].median(),  # Replace missing ages with median
        'anatom_site_general': 'unknown',             # Replace missing anatomical site with 'unknown'
        'sex': 'unknown'                              # Replace missing gender with 'unknown'
    }, inplace=True)

    # Merge metadata and ground truth on image ID
    combined_data = pd.merge(ground_truth, metadata, left_on='image', right_on='image')
    return combined_data


In [4]:
# Load and clean training and test datasets
train_data = load_and_clean_metadata(train_metadata_path, train_ground_truth_path)
test_data = load_and_clean_metadata(test_metadata_path, test_ground_truth_path)

In [5]:
train_data

,image,MEL,NV,BCC,AK,BKL,DF,VASC,SCC,UNK,age_approx,anatom_site_general,lesion_id,sex
0,ISIC_0000000,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,55.0,anterior torso,NaN,female
1,ISIC_0000001,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,30.0,anterior torso,NaN,female
2,ISIC_0000002,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,60.0,upper extremity,NaN,female
3,ISIC_0000003,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,30.0,upper extremity,NaN,male
4,ISIC_0000004,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,80.0,posterior torso,NaN,male
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25326,ISIC_0073247,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,85.0,head/neck,BCN_0003925,female
25327,ISIC_0073248,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,65.0,anterior torso,BCN_0001819,male
25328,ISIC_0073249,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,70.0,lower extremity,BCN_0001085,male
25329,ISIC_0073251,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,55.0,palms/soles,BCN_0002083,female


In [6]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25331 entries, 0 to 25330
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   image                25331 non-null  object 
 1   MEL                  25331 non-null  float64
 2   NV                   25331 non-null  float64
 3   BCC                  25331 non-null  float64
 4   AK                   25331 non-null  float64
 5   BKL                  25331 non-null  float64
 6   DF                   25331 non-null  float64
 7   VASC                 25331 non-null  float64
 8   SCC                  25331 non-null  float64
 9   UNK                  25331 non-null  float64
 10  age_approx           25331 non-null  float64
 11  anatom_site_general  25331 non-null  object 
 12  lesion_id            23247 non-null  object 
 13  sex                  25331 non-null  object 
dtypes: float64(10), object(4)
memory usage: 2.7+ MB


In [7]:
test_data

,image,MEL,NV,BCC,AK,BKL,DF,VASC,SCC,UNK,score_weight,validation_weight,age_approx,anatom_site_general,sex
0,ISIC_0034321,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,60.0,unknown,female
1,ISIC_0034322,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,70.0,anterior torso,male
2,ISIC_0034323,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,70.0,lower extremity,male
3,ISIC_0034324,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,70.0,lower extremity,male
4,ISIC_0034325,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,30.0,upper extremity,female
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8233,ISIC_0073236,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,75.0,anterior torso,male
8234,ISIC_0073243,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,65.0,lower extremity,male
8235,ISIC_0073250,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,30.0,anterior torso,female
8236,ISIC_0073252,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,85.0,head/neck,female


In [8]:
test_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8238 entries, 0 to 8237
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   image                8238 non-null   object 
 1   MEL                  8238 non-null   float64
 2   NV                   8238 non-null   float64
 3   BCC                  8238 non-null   float64
 4   AK                   8238 non-null   float64
 5   BKL                  8238 non-null   float64
 6   DF                   8238 non-null   float64
 7   VASC                 8238 non-null   float64
 8   SCC                  8238 non-null   float64
 9   UNK                  8238 non-null   float64
 10  score_weight         8238 non-null   float64
 11  validation_weight    8238 non-null   float64
 12  age_approx           8238 non-null   float64
 13  anatom_site_general  8238 non-null   object 
 14  sex                  8238 non-null   object 
dtypes: float64(12), object(3)
memory usage

In [9]:
# Add image path to check if images exist
train_data["image_path"] = train_data["image"].apply(lambda x: os.path.join(train_image_dir, x + ".jpg"))
test_data["image_path"] = test_data["image"].apply(lambda x: os.path.join(test_image_dir, x + ".jpg"))

# Check if all image paths exist
train_data["exists"] = train_data["image_path"].apply(lambda x: os.path.exists(x))
test_data["exists"] = test_data["image_path"].apply(lambda x: os.path.exists(x))



In [10]:
# Print the count of True/False for image existence
print("Training Data Image Existence Check:")
train_data["exists"].value_counts()  # Should be all True if paths are correct


Training Data Image Existence Check:


exists
True    25331
Name: count, dtype: int64

In [11]:
print("\nTesting Data Image Existence Check:")
test_data["exists"].value_counts()  # Should be all True if paths are correct


Testing Data Image Existence Check:


exists
True    8238
Name: count, dtype: int64

In [12]:
# Define transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),   # Resize to 224x224
    transforms.ToTensor(),           # Convert to tensor
    transforms.Normalize([0.5], [0.5])  # Normalize
])

In [13]:
class SkinDiseaseDataset(torch.utils.data.Dataset):
    def __init__(self, data, image_dir, transform=None):
        self.data = data
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # Get image ID and label
        image_id = self.data.iloc[idx]['image']
        label = self.data.iloc[idx][1:8].values.argmax()  # Assuming columns 1-7 are labels

        # Load image
        image_path = os.path.join(self.image_dir, f"{image_id}.jpg")
        image = Image.open(image_path).convert("RGB")

        # Apply transformations
        if self.transform:
            image = self.transform(image)

        return image, label

In [14]:
train_dataset = SkinDiseaseDataset(train_data, train_image_dir, transform)
test_dataset = SkinDiseaseDataset(test_data, test_image_dir, transform)


In [31]:
# Assuming you have a multi-core CPU

train_loader = torch.utils.data.DataLoader(
    train_dataset, 
    batch_size=64, 
    shuffle=False, 
    num_workers=4,# Set the number of worker processes
    pin_memory=True            # Optional: set to True if you're using a GPU
)

test_loader = torch.utils.data.DataLoader(
    test_dataset, 
    batch_size=64, 
    shuffle=False, 
    num_workers=4,# Set the number of worker processes
    pin_memory=True            # Optional: set to True if you're using a GPU
)


In [16]:
from PIL import Image

In [17]:
for images, labels in train_loader:
    print("Batch of images shape:", images.shape)
    print("Batch of labels shape:", labels.shape)
    break

Batch of images shape: torch.Size([128, 3, 224, 224])
Batch of labels shape: torch.Size([128])


In [20]:
# # Save data to a tensor (e.g., for images and labels)
# torch.save(train_dataset, "train_dataset.pth")
# torch.save(test_dataset, "test_dataset.pth")



In [21]:
# # Save the dataframes for training and test sets
# train_data.to_csv("train_data.csv", index=False)
# test_data.to_csv("test_data.csv", index=False)

In [22]:
# # Save the train and test loaders
# torch.save(train_loader, 'train_loader.pt')
# torch.save(test_loader, 'test_loader.pt')

In [19]:
# train_loader

In [18]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import models, transforms
from sklearn.metrics import accuracy_score


In [24]:
# # Load saved train and test loaders
# train_loader = torch.load('/kaggle/working/train_loader.pt')
# test_loader = torch.load('/kaggle/working/test_loader.pt')


<ipython-input-24-285730da7f19>:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  train_loader = torch.load('/kaggle/working/train_loader.pt')
<ipython-input-24-285730da7f19>

In [20]:
from transformers import ViTForImageClassification, ViTFeatureExtractor

# Load pre-trained ViT model
model = ViTForImageClassification.from_pretrained("google/vit-base-patch16-224-in21k", num_labels=7)  # Adjust num_labels

# Set the device (GPU or CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)


Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [21]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)


In [49]:
import torch

# Clear the GPU cache
torch.cuda.empty_cache()


In [50]:
from tqdm import tqdm

num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct_preds = 0
    total_preds = 0
    
    # Wrap the train_loader with tqdm for progress
    with tqdm(train_loader, unit="batch", desc=f"Epoch {epoch+1}/{num_epochs}") as tepoch:
        for images, labels in tepoch:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images).logits
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            correct_preds += (predicted == labels).sum().item()
            total_preds += labels.size(0)

            # Update progress bar description
            tepoch.set_postfix(loss=running_loss / (tepoch.n + 1), accuracy=correct_preds / total_preds)

    epoch_loss = running_loss / len(train_loader)
    epoch_accuracy = correct_preds / total_preds
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}, Accuracy: {epoch_accuracy:.4f}")
    
    # Save model after each epoch
    torch.save(model.state_dict(), f"model_epoch_{epoch+1}.pth")


Epoch 1/1: 100%|██████████| 9/9 [00:20<00:00,  2.24s/batch, accuracy=0.89, loss=0.326] 


Epoch [1/1], Loss: 0.3265, Accuracy: 0.8900


In [81]:
model.eval()
test_preds = []
test_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)  # Model output will likely be an ImageClassifierOutput

        # Access logits from the output
        logits = outputs.logits  # Make sure to use .logits here
        
        _, predicted = torch.max(logits, 1)  # Use logits to calculate predictions
        test_preds.extend(predicted.cpu().numpy())
        test_labels.extend(labels.cpu().numpy())

accuracy = accuracy_score(test_labels, test_preds)
print(f"Test Accuracy: {accuracy:.4f}")


Test Accuracy: 0.3100


In [ ]:
torch.save(model.state_dict(), 'final_model.pth')


In [70]:
import torch
from PIL import Image
from torchvision import transforms

# Assuming 'model' is your trained model and 'device' is either 'cuda' or 'cpu'

# Load your single image (adjust the path to the image you want to test with)
image_path = '/kaggle/input/isic-data/ISIC_2019_Test_Input/ISIC_2019_Test_Input/ISIC_0034383.jpg'
image = Image.open(image_path)

# Define the same transformations as used during training (resize, normalization, etc.)
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize to the input size expected by your model
    transforms.ToTensor(),          # Convert the image to a tensor
    transforms.Normalize([0.5], [0.5])  # Normalize as per your training setup
])

# Apply the transformations to the image
image = transform(image).unsqueeze(0)  # Add batch dimension (unsqueeze(0))



In [71]:
# Move the image tensor to the same device as the model
image = image.to(device)

# Set the model to evaluation mode
model.eval()

# Predict without computing gradients
with torch.no_grad():
    # Forward pass through the model
    outputs = model(image)
    
    # Access the logits from the outputs (for models like Hugging Face's)
    logits = outputs.logits  # Make sure to access .logits
    
    # Get the predicted class
    _, predicted = torch.max(logits, 1)

# Convert the predicted class index back to its label (if needed)
predicted_class = predicted.item()

# Output the predicted class
print(f"Predicted Class: {predicted_class}")


Predicted Class: 5
